# Adaptive Variable-Resolution 2.5D LiDAR Mapping
## Importance Engine + Adaptive Resolution Engine
**Team: Paradox Protocol | Environment: Google Colab + Python 3.x**

**Scope of THIS notebook:** ONLY the **Importance Engine** and the **Adaptive Resolution Engine**.
We do NOT implement semantic segmentation, the full 2.5D mapper, frontend / backend / database,
CARLA, ROS2, or any simulation system here. The code runs on **manually created region features**
today and is structured so **real LiDAR-derived features** can be connected later.

**Pipeline demonstrated:**
```
Input: RegionFeatures
        |
Normalization
        |
Importance Engine
        |
Importance score [0,1]
        |
Uncertainty-aware safety adjustment
        |
Resolution Engine
        |
5 cm / 10 cm / 20 cm / 50 cm
```

**Parameter status (read carefully):**
- Prototype engineering baseline = what we implement here (transparent defaults).
- Experimental optimization = tuning weights / thresholds on real data (NOT done here).
- Future research improvement = learned or adaptive policies (NOT claimed here).

We make no claims about real-time performance, optimality, or scientific novelty of the
weighted formula. All numbers below are computed live by the code.


In [ ]:
# Cell: Project Overview (executable check).
# This notebook implements ONLY: Importance Engine + Adaptive Resolution Engine.
import sys
print("Python:", sys.version)
import numpy
import matplotlib
print("NumPy:", numpy.__version__)
print("Matplotlib:", matplotlib.__version__)


## 2. Environment Setup
Verifies (and if needed, installs) `numpy` and `matplotlib`. On Colab both are pre-installed,
so this cell is normally a no-op check.


In [ ]:
# Cell: Environment Setup.
# On Colab, numpy + matplotlib are pre-installed. If imports ever fail, run:
# !pip install -q numpy matplotlib
import sys
print(sys.version)
try:
    import numpy as np
    import matplotlib.pyplot as plt
    print("NumPy", np.__version__, "| Matplotlib OK")
except ImportError as e:
    raise SystemExit(f"Missing dependency: {e}. Run: !pip install numpy matplotlib")


## 3. Imports
Standard library + NumPy + Matplotlib only (no seaborn, per the project spec).


In [ ]:
# Cell: Imports (standard-library + NumPy + Matplotlib only; no seaborn).
from __future__ import annotations  # modern type hints on older Pythons
import json                        # saving configuration
import math                        # finite / NaN / inf checks
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, Tuple, List
import numpy as np
import matplotlib.pyplot as plt
print("Imports OK")


## 4. Configuration
**Initial engineering parameters, NOT experimentally optimized values.**

The five weights sum to 1, so the base importance is a convex combination (weighted average)
of the five factors. Everything tunable lives in one frozen dataclass (`EngineConfig`), and
every constructor validates its inputs.


In [ ]:
# Cell: Configuration.
# NOTE: These are INITIAL ENGINEERING PARAMETERS, not experimentally optimized values.
# Status: prototype engineering baseline (see overview markdown).
"""Single place for every tunable knob of the two engines."""
BASE_WEIGHTS: Dict[str, float] = {
    "distance":    0.30,  # closeness of the region
    "semantic":    0.30,  # semantic class importance (e.g. pedestrian = 1.0)
    "terrain":     0.15,  # geometric / terrain complexity
    "dynamic":     0.15,  # motion / dynamic relevance
    "uncertainty": 0.10,  # perception uncertainty (also handled by safety bonus)
}

@dataclass(frozen=True)
class EngineConfig:
    """Every tunable knob of the two engines lives here."""
    max_distance_m: float = 100.0            # distance normalisation range
    weights: Dict[str, float] = field(default_factory=lambda: dict(BASE_WEIGHTS))
    uncertainty_bonus_weight: float = 0.15   # lambda in I_safe = min(1, I_base + lambda*U)
    # Resolution policy thresholds (initial experimental values, NOT claimed optimal):
    thresh_5cm: float = 0.75
    thresh_10cm: float = 0.50
    thresh_20cm: float = 0.25
    # Resolution values in metres:
    res_5cm_m: float = 0.05
    res_10cm_m: float = 0.10
    res_20cm_m: float = 0.20
    res_50cm_m: float = 0.50

    def __post_init__(self) -> None:
        # Validate max_distance:
        if not isinstance(self.max_distance_m, (int, float)) or not math.isfinite(self.max_distance_m):
            raise ValueError("max_distance_m must be a finite number")
        if self.max_distance_m <= 0:
            raise ValueError("max_distance_m must be > 0")
        # Validate uncertainty bonus (lambda):
        if not isinstance(self.uncertainty_bonus_weight, (int, float)) \
                or not math.isfinite(self.uncertainty_bonus_weight):
            raise ValueError("uncertainty_bonus_weight must be a finite number")
        if self.uncertainty_bonus_weight < 0:
            raise ValueError("uncertainty_bonus_weight must be >= 0")
        # Validate weights: all >= 0 and summing to 1:
        required = {"distance", "semantic", "terrain", "dynamic", "uncertainty"}
        if set(self.weights.keys()) != required:
            raise ValueError(f"weights must have exactly keys {required}, got {set(self.weights.keys())}")
        for key, value in self.weights.items():
            if not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"weight '{key}' must be a finite number, got {value!r}")
            if value < 0:
                raise ValueError(f"weight '{key}' must be >= 0, got {value}")
        total = sum(self.weights.values())
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"weights must sum to 1.0, got {total}")
        # Validate thresholds: strictly decreasing values in [0,1]:
        for name in ("thresh_5cm", "thresh_10cm", "thresh_20cm"):
            threshold = getattr(self, name)
            if not isinstance(threshold, (int, float)) or not math.isfinite(threshold) \
                    or not (0.0 <= threshold <= 1.0):
                raise ValueError(f"{name} must be a finite number in [0,1]")
        if not (self.thresh_5cm > self.thresh_10cm > self.thresh_20cm):
            raise ValueError("Need thresh_5cm > thresh_10cm > thresh_20cm")

CONFIG = EngineConfig()  # default baseline
print("Active configuration:")
print(CONFIG)
print("\nWeights sum:", sum(CONFIG.weights.values()))


## 5. Data Structures
`RawImportanceInput` is the region-level feature object the engines operate on.
`ImportanceResult` is the structured per-region output. Later, a thin converter function
(e.g. `features_from_lidar_cluster(...)`) can build `RawImportanceInput` from real
perception outputs without changing either engine.


In [ ]:
# Cell: Data Structures.
# The engines operate on a region-level feature object, so real LiDAR features
# can replace manual values later without changing the engines.
@dataclass(frozen=True)
class RawImportanceInput:
    """Region-level features for ONE map cell/region (manual today, LiDAR-derived later)."""
    distance_m: float          # >= 0 (may exceed max range; then clips to score 0)
    semantic_importance: float  # [0,1]
    terrain_complexity: float   # [0,1]
    dynamic_relevance: float    # [0,1]
    uncertainty: float          # [0,1]

    def __post_init__(self) -> None:
        # Type + finiteness checks first (catches NaN / inf with clear messages):
        for fname in ("distance_m", "semantic_importance", "terrain_complexity",
                      "dynamic_relevance", "uncertainty"):
            value = getattr(self, fname)
            if not isinstance(value, (int, float)):
                raise TypeError(f"{fname} must be a number, got {type(value).__name__}")
            if not math.isfinite(float(value)):
                raise ValueError(f"{fname} must be finite (NaN/inf not allowed), got {value!r}")
        # Domain checks:
        if float(self.distance_m) < 0:
            raise ValueError(f"distance_m cannot be negative, got {self.distance_m}")
        for fname in ("semantic_importance", "terrain_complexity",
                      "dynamic_relevance", "uncertainty"):
            value = float(getattr(self, fname))
            if not (0.0 <= value <= 1.0):
                raise ValueError(f"{fname} must be in [0,1], got {value}")

@dataclass(frozen=True)
class ImportanceResult:
    """Structured output of the full pipeline for one region."""
    distance_score: float       # D in [0,1]
    semantic_score: float       # S in [0,1]
    terrain_score: float        # T in [0,1]
    dynamic_score: float        # M in [0,1]
    uncertainty_score: float    # U in [0,1]
    base_importance: float      # I_base in [0,1]
    safe_importance: float      # I_safe in [0,1]
    selected_resolution_m: float  # e.g. 0.05
    resolution_level: str          # e.g. "5cm"

# Tiny smoke test:
demo = RawImportanceInput(distance_m=10.0, semantic_importance=1.0,
                          terrain_complexity=0.1, dynamic_relevance=1.0, uncertainty=0.05)
print("Data structures OK. Example input:", demo)


## 6. Input Validation
`RawImportanceInput` already validates itself in `__post_init__`. The helpers below let us
validate raw numbers *before* building the object, which is useful at the future LiDAR
interface boundary. Nothing is silently clipped or hidden here: bad inputs raise clear errors.


In [ ]:
# Cell: Input Validation.
# RawImportanceInput already validates in __post_init__; these helpers validate
# raw numbers BEFORE constructing the object (useful at the LiDAR boundary).
def validate_finite_number(value: float, name: str) -> float:
    """Ensure value is a finite real number; return it as float."""
    if not isinstance(value, (int, float)):
        raise TypeError(f"{name} must be a number, got {type(value).__name__}")
    result = float(value)
    if not math.isfinite(result):
        raise ValueError(f"{name} must be finite (NaN/inf not allowed), got {value!r}")
    return result

def validate_01(value: float, name: str) -> float:
    """Ensure value is a finite number inside [0,1]."""
    result = validate_finite_number(value, name)
    if not (0.0 <= result <= 1.0):
        raise ValueError(f"{name} must be in [0,1], got {result}")
    return result

def validate_distance(value: float, name: str = "distance_m") -> float:
    """Ensure distance is finite and >= 0 (beyond-max values allowed; they clip to 0)."""
    result = validate_finite_number(value, name)
    if result < 0:
        raise ValueError(f"{name} cannot be negative, got {result}")
    return result

def validate_region(region: RawImportanceInput) -> RawImportanceInput:
    """Re-validate an existing region object field-by-field (defensive check)."""
    validate_distance(region.distance_m, "distance_m")
    validate_01(region.semantic_importance, "semantic_importance")
    validate_01(region.terrain_complexity, "terrain_complexity")
    validate_01(region.dynamic_relevance, "dynamic_relevance")
    validate_01(region.uncertainty, "uncertainty")
    return region

print("Validators OK")
validate_region(demo)
print("demo region passes validation")


## 7. Normalization Functions
**Why normalize?** Raw features live on different scales (metres vs. unitless scores).
Mapping every factor to **[0,1]** makes the weighted sum meaningful and comparable across regions.

- Distance: `D = 1 - distance / max_distance`, clipped to [0,1] (0 m -> 1.0, 100 m -> 0.0).
- Everything else: reusable `normalize_01(value, min_value, max_value)`.


In [ ]:
# Cell: Normalization Functions.
# Why normalize? Raw features live on different scales (metres vs unitless
# scores). Mapping every factor to [0,1] makes the weighted sum meaningful.
def normalize_01(value: float, min_value: float = 0.0, max_value: float = 1.0) -> float:
    """Map [min_value, max_value] to [0,1] with clipping.

    Raises TypeError/ValueError for non-numeric, NaN/inf, or invalid bounds.
    Clips out-of-range *values* (does not raise) so noisy inputs survive.
    """
    if not isinstance(min_value, (int, float)) or not isinstance(max_value, (int, float)):
        raise TypeError("min_value and max_value must be numbers")
    if not math.isfinite(min_value) or not math.isfinite(max_value):
        raise ValueError("min_value and max_value must be finite")
    if max_value <= min_value:
        raise ValueError(f"max_value ({max_value}) must be > min_value ({min_value})")
    raw = validate_finite_number(value, "value")
    scaled = (raw - float(min_value)) / (float(max_value) - float(min_value))
    return float(max(0.0, min(1.0, scaled)))

def normalize_distance(distance_m: float, max_distance_m: float = 100.0) -> float:
    """Normalise distance: 0 m -> 1.0, max_distance_m -> 0.0, beyond -> 0.0 (clipped).

    Rejects negative / NaN / inf distances with clear errors.
    """
    dist = validate_distance(distance_m, "distance_m")
    if not isinstance(max_distance_m, (int, float)) or not math.isfinite(max_distance_m):
        raise ValueError("max_distance_m must be a finite number")
    if max_distance_m <= 0:
        raise ValueError("max_distance_m must be > 0")
    return float(max(0.0, min(1.0, 1.0 - dist / float(max_distance_m))))

# Quick demo:
for d in [0, 10, 50, 100, 150]:
    print(f"distance {d:>6} m -> score {normalize_distance(d, 100.0):.3f}")


## 8. Importance Engine (base formula)
`I_base = wd*D + ws*S + wt*T + wm*M + wu*U`, clipped to [0,1], where
D = normalized closeness (near = important), S = semantic importance,
T = terrain complexity, M = dynamic relevance, U = uncertainty.


In [ ]:
# Cell: Importance Engine (base formula).
# I_base = wd*D + ws*S + wt*T + wm*M + wu*U, clipped to [0,1].
# D = closeness, S = semantics, T = terrain, M = dynamics, U = uncertainty.
def compute_base_importance(distance_score: float, semantic: float, terrain: float,
                            dynamic: float, uncertainty: float, config: EngineConfig) -> float:
    """Weighted sum of the five [0,1] factors, clipped to [0,1]."""
    dist = validate_01(distance_score, "distance_score")
    sem = validate_01(semantic, "semantic")
    ter = validate_01(terrain, "terrain")
    dyn = validate_01(dynamic, "dynamic")
    unc = validate_01(uncertainty, "uncertainty")
    w = config.weights
    raw = (w["distance"] * dist + w["semantic"] * sem + w["terrain"] * ter
           + w["dynamic"] * dyn + w["uncertainty"] * unc)
    return float(max(0.0, min(1.0, raw)))

print("Base formula check:",
      compute_base_importance(0.9, 1.0, 0.1, 1.0, 0.05, CONFIG))  # ~0.74


## 9. Resolution Engine
Initial experimental policy (**not claimed optimal**). Boundaries use `>=`, so the exact
values 0.75 / 0.50 / 0.25 fall into the *finer* bin:

```
Importance >= 0.75 -> 0.05 m (5 cm) | Importance >= 0.50 -> 0.10 m (10 cm)
Importance >= 0.25 -> 0.20 m (20 cm) | Importance <  0.25 -> 0.50 m (50 cm)
```


In [ ]:
# Cell: Resolution Engine.
# Initial experimental policy (NOT claimed optimal). Boundaries use >= so
# 0.75 / 0.50 / 0.25 fall into the finer bin.
class ResolutionEngine:
    """Maps an importance score in [0,1] to (resolution_m, level_label)."""
    def __init__(self, config: EngineConfig) -> None:
        self.config = config  # keeps thresholds/values in one place

    def select(self, importance: float) -> Tuple[float, str]:
        f = validate_finite_number(importance, "importance")
        if not (0.0 <= f <= 1.0):
            raise ValueError(f"importance must be in [0,1], got {f}")
        cfg = self.config
        if f >= cfg.thresh_5cm:
            return (cfg.res_5cm_m, "5cm")
        if f >= cfg.thresh_10cm:
            return (cfg.res_10cm_m, "10cm")
        if f >= cfg.thresh_20cm:
            return (cfg.res_20cm_m, "20cm")
        return (cfg.res_50cm_m, "50cm")

print("Resolution check:", ResolutionEngine(CONFIG).select(0.75))  # (0.05, '5cm')


## 10. Uncertainty-Aware Safety Modifier
`I_safe = min(1, I_base + lambda * U)` with `lambda = 0.15` by default.

- `I_base` is the normal environmental importance.
- Uncertainty can **promote** a region to a finer resolution: we would rather spend a
  little extra compute than lose potentially important information when perception is unsure.
- The bonus is configurable (`uncertainty_bonus_weight`), so we can later A/B test whether
  it helps. Setting `lambda = 0` disables it cleanly.


In [ ]:
# Cell: Uncertainty-Aware Safety Modifier.
# I_safe = min(1, I_base + lambda*U), lambda = 0.15 by default.
# I_base = normal environmental importance; uncertainty can PROMOTE a region
# to finer resolution so we never lose possibly-important information just
# because perception was unsure. Set lambda = 0 to disable (for A/B tests).
def apply_safety_modifier(base_importance: float, uncertainty: float,
                          config: EngineConfig) -> float:
    """I_safe = min(1, I_base + lambda*U). Both inputs must be in [0,1]."""
    base = validate_01(base_importance, "base_importance")
    unc = validate_01(uncertainty, "uncertainty")
    bonus = validate_finite_number(config.uncertainty_bonus_weight, "uncertainty_bonus_weight")
    if bonus < 0:
        raise ValueError("uncertainty_bonus_weight must be >= 0")
    return float(min(1.0, base + bonus * unc))

# Demo: same base importance, higher uncertainty -> higher safe importance:
for u in [0.0, 0.1, 0.5, 0.9, 1.0]:
    print(f"U={u:.1f} -> I_safe={apply_safety_modifier(0.50, u, CONFIG):.4f}")


## 11. Complete Region Processing
The full pipeline in one user-facing call: `engine.process(region)`.

```
RawImportanceInput -> validation -> distance normalization -> other normalizations
-> base importance -> uncertainty-aware importance -> resolution -> ImportanceResult
```


In [ ]:
# Cell: Complete Region Processing.
# Full pipeline: validate -> normalize -> base -> safety -> resolution.
# User-facing API is one line: engine.process(region).
class ImportanceEngine:
    """Facade combining normalization + base importance + safety + resolution."""
    def __init__(self, config: EngineConfig = EngineConfig()) -> None:
        self.config = config
        self.resolution_engine = ResolutionEngine(config)

    def process(self, region: RawImportanceInput) -> ImportanceResult:
        # 1) validate raw inputs (raises clear errors; never silently hides them)
        validate_region(region)
        # 2) normalize each factor to [0,1]
        dist_score = normalize_distance(region.distance_m, self.config.max_distance_m)
        sem = normalize_01(region.semantic_importance, 0.0, 1.0)
        ter = normalize_01(region.terrain_complexity, 0.0, 1.0)
        dyn = normalize_01(region.dynamic_relevance, 0.0, 1.0)
        unc = normalize_01(region.uncertainty, 0.0, 1.0)
        # 3) base importance (weighted average)
        base = compute_base_importance(dist_score, sem, ter, dyn, unc, self.config)
        # 4) uncertainty-aware safety adjustment
        safe = apply_safety_modifier(base, unc, self.config)
        # 5) resolution selection uses the SAFE score
        res_m, level = self.resolution_engine.select(safe)
        return ImportanceResult(distance_score=dist_score, semantic_score=sem,
                                terrain_score=ter, dynamic_score=dyn, uncertainty_score=unc,
                                base_importance=base, safe_importance=safe,
                                selected_resolution_m=res_m, resolution_level=level)

engine = ImportanceEngine(CONFIG)
print("Pipeline check:", engine.process(demo))


## 12. Unit Tests
Executable checks for normalization values, importance ranges, resolution boundaries,
weight validation, and data validation (negative distance, out-of-range scores, NaN, inf).
Each test prints PASS/FAIL.


In [ ]:
# Cell: Unit Tests (normalization, ranges, boundaries, weights, data validation).
passed, failed = 0, 0

def check(name: str, fn) -> None:
    """Run fn; count PASS unless it raises."""
    global passed, failed
    try:
        fn()
        print(f"[PASS] {name}")
        passed += 1
    except AssertionError as e:
        print(f"[FAIL] {name}: assertion: {e}")
        failed += 1
    except Exception as e:
        print(f"[FAIL] {name}: unexpected {type(e).__name__}: {e}")
        failed += 1

def expect_error(name: str, fn, exc=(ValueError, TypeError)) -> None:
    """Pass only if fn raises ValueError/TypeError."""
    global passed, failed
    try:
        fn()
        print(f"[FAIL] {name}: expected error but nothing raised")
        failed += 1
    except exc as e:
        print(f"[PASS] {name} (raised {type(e).__name__}: {e})")
        passed += 1
    except Exception as e:
        print(f"[FAIL] {name}: wrong exception {type(e).__name__}: {e}")
        failed += 1

def _assert_equal(actual, expected):
    if actual != expected:
        raise AssertionError(f"got {actual!r}, expected {expected!r}")

# --- Normalization ---
check("dist 0 -> 1.0", lambda: _assert_equal(normalize_distance(0), 1.0))
check("dist 10 -> 0.9", lambda: _assert_equal(normalize_distance(10), 0.9))
check("dist 50 -> 0.5", lambda: _assert_equal(normalize_distance(50), 0.5))
check("dist 100 -> 0.0", lambda: _assert_equal(normalize_distance(100), 0.0))
check("dist 150 clips to 0.0", lambda: _assert_equal(normalize_distance(150), 0.0))
expect_error("negative distance raises", lambda: normalize_distance(-5))
check("normalize_01 mid", lambda: _assert_equal(normalize_01(0.5), 0.5))
expect_error("normalize_01 bad bounds raise", lambda: normalize_01(0.5, 1.0, 1.0))

# --- Importance ranges ---
def _ranges():
    eng = ImportanceEngine()
    for region in [RawImportanceInput(0, 0, 0, 0, 0),
                   RawImportanceInput(100, 1, 1, 1, 1),
                   RawImportanceInput(37.5, 0.4, 0.6, 0.2, 0.8)]:
        out = eng.process(region)
        assert 0.0 <= out.base_importance <= 1.0, out
        assert 0.0 <= out.safe_importance <= 1.0, out
check("importance always in [0,1]", _ranges)

# --- Resolution boundaries ---
_BOUNDS = {0.95: "5cm", 0.75: "5cm", 0.74: "10cm", 0.50: "10cm", 0.49: "20cm",
           0.25: "20cm", 0.24: "50cm", 0.0: "50cm", 1.0: "5cm"}
def _bounds():
    resolver = ResolutionEngine(EngineConfig())
    for value, level in _BOUNDS.items():
        got = resolver.select(value)[1]
        if got != level:
            raise AssertionError(f"importance {value}: got {got}, expected {level}")
check("resolution boundaries", _bounds)

# --- Weight validation ---
expect_error("negative weight raises", lambda: EngineConfig(weights={
    "distance": -0.1, "semantic": 0.5, "terrain": 0.2, "dynamic": 0.2, "uncertainty": 0.2}))
expect_error("weights summing to != 1 raise", lambda: EngineConfig(weights={
    "distance": 0.5, "semantic": 0.5, "terrain": 0.5, "dynamic": 0.5, "uncertainty": 0.5}))

# --- Data validation ---
expect_error("negative distance raises", lambda: RawImportanceInput(-1, 0.5, 0.5, 0.5, 0.5))
expect_error("semantic > 1 raises", lambda: RawImportanceInput(10, 1.5, 0.5, 0.5, 0.5))
expect_error("terrain < 0 raises", lambda: RawImportanceInput(10, 0.5, -0.1, 0.5, 0.5))
expect_error("uncertainty > 1 raises", lambda: RawImportanceInput(10, 0.5, 0.5, 0.5, 1.2))
expect_error("NaN raises", lambda: RawImportanceInput(float("nan"), 0.5, 0.5, 0.5, 0.5))
expect_error("inf raises", lambda: RawImportanceInput(float("inf"), 0.5, 0.5, 0.5, 0.5))
print(f"\nUnit tests: {passed} passed, {failed} failed")
assert failed == 0, "Fix failing unit tests before continuing"


## 13. Scenario Tests
Four realistic regions. Expected importance values are **calculated by the implementation**,
never hard-coded.


In [ ]:
# Cell: Scenario Tests (values calculated by the implementation, never hard-coded).
scenarios = {
    "A - Nearby pedestrian":         RawImportanceInput(10, 1.0, 0.10, 1.0, 0.05),
    "B - Far pedestrian":            RawImportanceInput(70, 1.0, 0.10, 1.0, 0.05),
    "C - Empty distant road":        RawImportanceInput(70, 0.1, 0.05, 0.0, 0.05),
    "D - Unknown possible obstacle": RawImportanceInput(60, 0.6, 0.70, 0.4, 0.90),
}
print(f"{'Scenario':<32}{'Dist':>7}  {'Base':>6}  {'Safe':>6}  Resolution")
print("-" * 70)
results = {}
for name, region in scenarios.items():
    out = engine.process(region)
    results[name] = out
    print(f"{name:<32}{region.distance_m:>6.1f}m  {out.base_importance:>6.3f}  "
          f"{out.safe_importance:>6.3f}  {out.selected_resolution_m:.2f} m ({out.resolution_level})")


## 13b. Most Important Conceptual Test
Pedestrian at 70 m vs. empty road at 70 m. Distance is identical, so any difference in the
output must come from semantic / context features. This is the core reason the system uses
more than distance alone.


In [ ]:
# Cell: Conceptual Test - same distance, different meaning.
# Pedestrian at 70 m vs empty road at 70 m. Distance is identical, so any
# output difference must come from semantic/context features.
ped = RawImportanceInput(distance_m=70, semantic_importance=1.0, terrain_complexity=0.1,
                         dynamic_relevance=1.0, uncertainty=0.05)
road = RawImportanceInput(distance_m=70, semantic_importance=0.1, terrain_complexity=0.05,
                          dynamic_relevance=0.0, uncertainty=0.05)
r_ped, r_road = engine.process(ped), engine.process(road)
print(f"Pedestrian: base={r_ped.base_importance:.4f} safe={r_ped.safe_importance:.4f} -> "
      f"{r_ped.selected_resolution_m:.2f} m ({r_ped.resolution_level})")
print(f"Empty road: base={r_road.base_importance:.4f} safe={r_road.safe_importance:.4f} -> "
      f"{r_road.selected_resolution_m:.2f} m ({r_road.resolution_level})")
print()
print("Both regions are 70 m away.")
print("The pedestrian receives higher importance because the system considers "
      "semantic and dynamic relevance, not just distance.")
assert r_ped.safe_importance > r_road.safe_importance, "Conceptual test failed!"
print("Conceptual test PASSED: meaning changed the resolution decision.")


## 14. Sensitivity / Weight Analysis
**Sensitivity analysis / initial parameter study** - not a claim about the best configuration.
We run the same regions through distance-heavy, semantic-heavy, and balanced weightings and
observe how the outputs move.


In [ ]:
# Cell: Sensitivity / Weight Analysis.
# SENSITIVITY ANALYSIS / INITIAL PARAMETER STUDY - not a claim about the best
# configuration. Same regions, three weightings; observe how outputs move.
configs = {
    "A: distance-heavy": EngineConfig(weights={
        "distance": 0.60, "semantic": 0.15, "terrain": 0.10, "dynamic": 0.10, "uncertainty": 0.05}),
    "B: semantic-heavy": EngineConfig(weights={
        "distance": 0.15, "semantic": 0.60, "terrain": 0.05, "dynamic": 0.15, "uncertainty": 0.05}),
    "C: balanced":       EngineConfig(),  # the default baseline
}
test_regions = {
    "near pedestrian": RawImportanceInput(10, 1.0, 0.1, 1.0, 0.05),
    "far pedestrian":  RawImportanceInput(70, 1.0, 0.1, 1.0, 0.05),
    "empty road":      RawImportanceInput(70, 0.1, 0.05, 0.0, 0.05),
}
print(f"{'Region':<16}" + "".join(f"{key:>22}" for key in configs))
for region_name, region in test_regions.items():
    row = f"{region_name:<16}"
    for cfg_name, cfg in configs.items():
        out = ImportanceEngine(cfg).process(region)
        row += f"{out.safe_importance:>22.3f}"
    print(row)
print("\nNote: sensitivity study only. We do NOT claim any configuration is best")
print("unless a controlled experiment on real data demonstrates it.")


## 15. Visualization
Four plain-Matplotlib figures (no seaborn): the distance curve, the importance-to-resolution
step function, the scenario comparison, and a single-feature sweep.


In [ ]:
# Cell: Visualization (plain Matplotlib only; no seaborn).
# Plot 1: Distance vs normalized distance score.
ds = np.linspace(0, 150, 301)
scores = [normalize_distance(float(d), CONFIG.max_distance_m) for d in ds]
plt.figure()
plt.plot(ds, scores)
plt.axvline(CONFIG.max_distance_m, linestyle="--")
plt.title("Plot 1 - Distance vs Normalized Distance Score")
plt.xlabel("Distance (m)")
plt.ylabel("Distance score D in [0,1]")
plt.grid(True)
plt.show()

# Plot 2: Importance score vs selected resolution (step function).
imps = np.linspace(0, 1, 401)
res_vals = [ResolutionEngine(CONFIG).select(float(i))[0] for i in imps]
plt.figure()
plt.step(imps, res_vals, where="post")
plt.title("Plot 2 - Importance Score vs Selected Resolution")
plt.xlabel("Safe importance in [0,1]")
plt.ylabel("Resolution (m)")
plt.grid(True)
plt.show()

# Plot 3: Scenario comparison (base vs safe importance).
names = list(results.keys())
short = ["A: near ped", "B: far ped", "C: empty road", "D: unknown"]
base_v = [results[k].base_importance for k in names]
safe_v = [results[k].safe_importance for k in names]
x = np.arange(len(short))
w = 0.35
plt.figure()
plt.bar(x - w / 2, base_v, width=w, label="base")
plt.bar(x + w / 2, safe_v, width=w, label="safe")
plt.xticks(x, short)
plt.title("Plot 3 - Scenario Comparison (Base vs Safe Importance)")
plt.ylabel("Importance in [0,1]")
plt.legend()
plt.grid(True, axis="y")
plt.show()

# Plot 4: Sweep ONE feature (semantic) while holding others constant.
sem = np.linspace(0, 1, 101)
fixed = dict(distance_m=50, terrain_complexity=0.2, dynamic_relevance=0.3, uncertainty=0.1)
final = [engine.process(RawImportanceInput(
    fixed["distance_m"], float(s), fixed["terrain_complexity"],
    fixed["dynamic_relevance"], fixed["uncertainty"])).safe_importance for s in sem]
plt.figure()
plt.plot(sem, final)
plt.title("Plot 4 - Semantic Importance -> Final (Safe) Importance\n(other features fixed)")
plt.xlabel("Semantic importance S")
plt.ylabel("Safe importance")
plt.grid(True)
plt.show()


## 15b. Edge Cases
Degenerate inputs (0 m, 100 m, beyond max range, min/max uncertainty, all-zeros, all-ones).
Every output must remain in its valid range.


In [ ]:
# Cell: Edge Cases (every output must stay in valid ranges).
edge_cases = {
    "object at 0 m":       RawImportanceInput(0, 0.5, 0.5, 0.5, 0.5),
    "object at 100 m":     RawImportanceInput(100, 0.5, 0.5, 0.5, 0.5),
    "beyond max range":    RawImportanceInput(250, 0.5, 0.5, 0.5, 0.5),
    "maximum uncertainty": RawImportanceInput(30, 0.3, 0.3, 0.3, 1.0),
    "zero uncertainty":    RawImportanceInput(30, 0.3, 0.3, 0.3, 0.0),
    "all features = 0":    RawImportanceInput(100, 0.0, 0.0, 0.0, 0.0),
    "all features = 1":    RawImportanceInput(0, 1.0, 1.0, 1.0, 1.0),
}
for name, region in edge_cases.items():
    out = engine.process(region)
    assert 0.0 <= out.base_importance <= 1.0 and 0.0 <= out.safe_importance <= 1.0
    assert out.selected_resolution_m in (0.05, 0.10, 0.20, 0.50)
    print(f"{name:<20} base={out.base_importance:.3f} safe={out.safe_importance:.3f} "
          f"-> {out.selected_resolution_m:.2f} m ({out.resolution_level})")
print("\nAll edge cases handled within valid ranges.")


## 16. Save Configuration
Persists the baseline as `config/importance_config.json` using `pathlib`
(the directory is created automatically).


In [ ]:
# Cell: Save Configuration (pathlib creates the directory automatically).
config_path = Path("config/importance_config.json")
config_path.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "max_distance_m": CONFIG.max_distance_m,
    "weights": dict(CONFIG.weights),
    "uncertainty_bonus_weight": CONFIG.uncertainty_bonus_weight,
    "resolution_thresholds": {
        "gte_5cm": CONFIG.thresh_5cm,
        "gte_10cm": CONFIG.thresh_10cm,
        "gte_20cm": CONFIG.thresh_20cm,
        "else_50cm": True,
    },
    "resolution_values_m": {
        "5cm": CONFIG.res_5cm_m,
        "10cm": CONFIG.res_10cm_m,
        "20cm": CONFIG.res_20cm_m,
        "50cm": CONFIG.res_50cm_m,
    },
    "notes": "Initial engineering baseline - not experimentally optimized.",
}
config_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved config to {config_path.resolve()}")
print(config_path.read_text(encoding="utf-8"))


## 17. Save Reusable Module Files
Writes the **real implementation** (not placeholders) into `src/`:
`importance_types.py`, `resolution_engine.py`, `importance_engine.py`.
The file contents are embedded below as base64 (so quoting can never break),
then decoded, written, imported, and cross-checked against the notebook implementation.


In [ ]:
# Cell: Save Reusable Module Files (real implementation, not placeholders).
# Each SRC_* string below holds the COMPLETE file content. Run this cell to
# write src/importance_types.py, src/resolution_engine.py, src/importance_engine.py,
# then import them and cross-check them against the notebook implementation.
SRC_DIR = Path("src")
SRC_DIR.mkdir(parents=True, exist_ok=True)
SRC_TYPES = """
'''Reusable types + EngineConfig for the Importance / Resolution engines.

Status: prototype engineering baseline (initial parameters, not optimized).
'''
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict

# Initial engineering weights (must sum to 1). NOT experimentally optimized.
BASE_WEIGHTS: Dict[str, float] = {
    "distance": 0.30,    # closeness of the region
    "semantic": 0.30,    # semantic class importance (e.g. pedestrian = 1.0)
    "terrain": 0.15,     # geometric / terrain complexity
    "dynamic": 0.15,     # motion / dynamic relevance
    "uncertainty": 0.10,  # perception uncertainty
}


@dataclass(frozen=True)
class EngineConfig:
    '''Every tunable knob of the two engines lives here.'''

    max_distance_m: float = 100.0
    weights: Dict[str, float] = field(default_factory=lambda: dict(BASE_WEIGHTS))
    uncertainty_bonus_weight: float = 0.15  # lambda in I_safe = min(1, I_base + lambda*U)
    # Resolution thresholds: initial experimental values, NOT claimed optimal.
    thresh_5cm: float = 0.75
    thresh_10cm: float = 0.50
    thresh_20cm: float = 0.25
    # Resolution values in metres.
    res_5cm_m: float = 0.05
    res_10cm_m: float = 0.10
    res_20cm_m: float = 0.20
    res_50cm_m: float = 0.50

    def __post_init__(self) -> None:
        if not isinstance(self.max_distance_m, (int, float)) or not math.isfinite(self.max_distance_m):
            raise ValueError("max_distance_m must be a finite number")
        if self.max_distance_m <= 0:
            raise ValueError("max_distance_m must be > 0")
        if not isinstance(self.uncertainty_bonus_weight, (int, float)):
            raise ValueError("uncertainty_bonus_weight must be a number")
        if not math.isfinite(self.uncertainty_bonus_weight) or self.uncertainty_bonus_weight < 0:
            raise ValueError("uncertainty_bonus_weight must be finite and >= 0")
        required = {"distance", "semantic", "terrain", "dynamic", "uncertainty"}
        if set(self.weights.keys()) != required:
            raise ValueError(f"weights must have exactly keys {required}")
        for key, value in self.weights.items():
            if not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"weight '{key}' must be a finite number, got {value!r}")
            if value < 0:
                raise ValueError(f"weight '{key}' must be >= 0, got {value}")
        total = sum(self.weights.values())
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"weights must sum to 1.0, got {total}")
        for name in ("thresh_5cm", "thresh_10cm", "thresh_20cm"):
            threshold = getattr(self, name)
            if not isinstance(threshold, (int, float)) or not math.isfinite(threshold):
                raise ValueError(f"{name} must be a finite number")
            if not (0.0 <= threshold <= 1.0):
                raise ValueError(f"{name} must be in [0,1]")
        if not (self.thresh_5cm > self.thresh_10cm > self.thresh_20cm):
            raise ValueError("Need thresh_5cm > thresh_10cm > thresh_20cm")


@dataclass(frozen=True)
class RawImportanceInput:
    '''Region-level features for ONE map cell/region.

    Manual values today; LiDAR-derived features later. The engines only
    depend on this object, never on a specific dataset.
    '''

    distance_m: float  # >= 0 (may exceed max range; then clips to score 0)
    semantic_importance: float  # [0,1]
    terrain_complexity: float  # [0,1]
    dynamic_relevance: float  # [0,1]
    uncertainty: float  # [0,1]

    def __post_init__(self) -> None:
        for fname in ("distance_m", "semantic_importance", "terrain_complexity",
                      "dynamic_relevance", "uncertainty"):
            value = getattr(self, fname)
            if not isinstance(value, (int, float)):
                raise TypeError(f"{fname} must be a number, got {type(value).__name__}")
            if not math.isfinite(float(value)):
                raise ValueError(f"{fname} must be finite (NaN/inf not allowed), got {value!r}")
        if float(self.distance_m) < 0:
            raise ValueError(f"distance_m cannot be negative, got {self.distance_m}")
        for fname in ("semantic_importance", "terrain_complexity",
                      "dynamic_relevance", "uncertainty"):
            value = float(getattr(self, fname))
            if not (0.0 <= value <= 1.0):
                raise ValueError(f"{fname} must be in [0,1], got {value}")


@dataclass(frozen=True)
class ImportanceResult:
    '''Structured pipeline output for one region.'''

    distance_score: float
    semantic_score: float
    terrain_score: float
    dynamic_score: float
    uncertainty_score: float
    base_importance: float
    safe_importance: float
    selected_resolution_m: float
    resolution_level: str
"""
SRC_RESOLUTION = """
'''ResolutionEngine: importance score in [0,1] -> (resolution metres, label).

Initial experimental policy (NOT claimed optimal):
    I >= 0.75 -> 0.05 m (5cm) | I >= 0.50 -> 0.10 m (10cm)
    I >= 0.25 -> 0.20 m (20cm) | I <  0.25 -> 0.50 m (50cm)
Boundaries use >=, so 0.75 / 0.50 / 0.25 fall into the finer bin.
'''
from __future__ import annotations

import math
from typing import Tuple


class ResolutionEngine:
    '''Maps an importance score to a concrete map resolution.'''

    def __init__(self, config) -> None:
        # `config` is an EngineConfig (duck-typed to avoid a hard import cycle).
        self.config = config

    def select(self, importance: float) -> Tuple[float, str]:
        '''Return (resolution_m, level_label) for an importance in [0,1].'''
        if not isinstance(importance, (int, float)) or not math.isfinite(importance):
            raise ValueError(f"importance must be a finite number, got {importance!r}")
        score = float(importance)
        if not (0.0 <= score <= 1.0):
            raise ValueError(f"importance must be in [0,1], got {score}")
        cfg = self.config
        if score >= cfg.thresh_5cm:
            return (cfg.res_5cm_m, "5cm")
        if score >= cfg.thresh_10cm:
            return (cfg.res_10cm_m, "10cm")
        if score >= cfg.thresh_20cm:
            return (cfg.res_20cm_m, "20cm")
        return (cfg.res_50cm_m, "50cm")
"""
SRC_IMPORTANCE = """
'''ImportanceEngine: normalization + base importance + safety + resolution.

Pipeline per region:
    RawImportanceInput -> validation -> normalization -> I_base
    -> I_safe = min(1, I_base + lambda*U) -> resolution -> ImportanceResult
'''
from __future__ import annotations

import math
from typing import Dict

from importance_types import EngineConfig, ImportanceResult, RawImportanceInput
from resolution_engine import ResolutionEngine


def validate_finite_number(value: float, name: str) -> float:
    '''Ensure value is a finite real number; return it as float.'''
    if not isinstance(value, (int, float)):
        raise TypeError(f"{name} must be a number, got {type(value).__name__}")
    result = float(value)
    if not math.isfinite(result):
        raise ValueError(f"{name} must be finite (NaN/inf not allowed), got {value!r}")
    return result


def validate_01(value: float, name: str) -> float:
    '''Ensure value is a finite number inside [0,1].'''
    result = validate_finite_number(value, name)
    if not (0.0 <= result <= 1.0):
        raise ValueError(f"{name} must be in [0,1], got {result}")
    return result


def normalize_01(value: float, min_value: float = 0.0, max_value: float = 1.0) -> float:
    '''Map [min_value, max_value] to [0,1] with clipping.

    Raises for non-numeric / NaN / inf inputs and for invalid bounds
    (bounds must be finite with max_value > min_value). Out-of-range
    *values* are clipped, not rejected, so noisy inputs survive.
    '''
    if not isinstance(min_value, (int, float)) or not isinstance(max_value, (int, float)):
        raise TypeError("min_value and max_value must be numbers")
    if not math.isfinite(min_value) or not math.isfinite(max_value):
        raise ValueError("min_value and max_value must be finite")
    if max_value <= min_value:
        raise ValueError(f"max_value ({max_value}) must be > min_value ({min_value})")
    raw = validate_finite_number(value, "value")
    scaled = (raw - float(min_value)) / (float(max_value) - float(min_value))
    return float(max(0.0, min(1.0, scaled)))


def normalize_distance(distance_m: float, max_distance_m: float = 100.0) -> float:
    '''Normalise distance: 0 m -> 1.0, max_distance_m -> 0.0, beyond -> 0.0.

    Rejects negative / NaN / inf distances with clear errors.
    '''
    distance = validate_finite_number(distance_m, "distance_m")
    if distance < 0:
        raise ValueError(f"distance_m cannot be negative, got {distance}")
    if not isinstance(max_distance_m, (int, float)) or not math.isfinite(max_distance_m):
        raise ValueError("max_distance_m must be a finite number")
    if max_distance_m <= 0:
        raise ValueError("max_distance_m must be > 0")
    return float(max(0.0, min(1.0, 1.0 - distance / float(max_distance_m))))


def compute_base_importance(distance_score: float, semantic: float, terrain: float,
                            dynamic: float, uncertainty: float,
                            config: EngineConfig) -> float:
    '''Weighted average I_base = wd*D + ws*S + wt*T + wm*M + wu*U, clipped to [0,1].'''
    weights: Dict[str, float] = config.weights
    raw = (weights["distance"] * validate_01(distance_score, "distance_score")
           + weights["semantic"] * validate_01(semantic, "semantic")
           + weights["terrain"] * validate_01(terrain, "terrain")
           + weights["dynamic"] * validate_01(dynamic, "dynamic")
           + weights["uncertainty"] * validate_01(uncertainty, "uncertainty"))
    return float(max(0.0, min(1.0, raw)))


def apply_safety_modifier(base_importance: float, uncertainty: float,
                          config: EngineConfig) -> float:
    '''Safety-aware score: I_safe = min(1, I_base + lambda * U).

    Lets uncertain regions step up to finer resolution instead of risking
    lost information. Set lambda = 0 to disable the bonus for A/B tests.
    '''
    base = validate_01(base_importance, "base_importance")
    uncertain = validate_01(uncertainty, "uncertainty")
    bonus = validate_finite_number(config.uncertainty_bonus_weight, "uncertainty_bonus_weight")
    if bonus < 0:
        raise ValueError("uncertainty_bonus_weight must be >= 0")
    return float(min(1.0, base + bonus * uncertain))


class ImportanceEngine:
    '''User-facing facade: engine.process(region) -> ImportanceResult.'''

    def __init__(self, config: EngineConfig = EngineConfig()) -> None:
        self.config = config
        self.resolution_engine = ResolutionEngine(config)

    def process(self, region: RawImportanceInput) -> ImportanceResult:
        '''Run the full pipeline for one region (raises clear errors, never silent).'''
        # RawImportanceInput.__post_init__ already validated; re-check defensively
        # via normalize_* (they raise the same clear errors at the boundary).
        distance_score = normalize_distance(region.distance_m, self.config.max_distance_m)
        semantic = normalize_01(region.semantic_importance, 0.0, 1.0)
        terrain = normalize_01(region.terrain_complexity, 0.0, 1.0)
        dynamic = normalize_01(region.dynamic_relevance, 0.0, 1.0)
        uncertainty = normalize_01(region.uncertainty, 0.0, 1.0)
        base = compute_base_importance(distance_score, semantic, terrain,
                                       dynamic, uncertainty, self.config)
        safe = apply_safety_modifier(base, uncertainty, self.config)
        resolution_m, level = self.resolution_engine.select(safe)
        return ImportanceResult(distance_score, semantic, terrain, dynamic,
                                uncertainty, base, safe, resolution_m, level)
"""
(SRC_DIR / "importance_types.py").write_text(SRC_TYPES, encoding="utf-8")
(SRC_DIR / "resolution_engine.py").write_text(SRC_RESOLUTION, encoding="utf-8")
(SRC_DIR / "importance_engine.py").write_text(SRC_IMPORTANCE, encoding="utf-8")
print("Wrote:", sorted(str(p) for p in SRC_DIR.glob("*.py")))
# Prove the file modules import and agree with the notebook implementation:
import sys as _sys
_sys.path.insert(0, str(SRC_DIR.resolve()))
from importance_types import RawImportanceInput as FileInput, EngineConfig as FileConfig
from importance_engine import ImportanceEngine as FileEngine
file_engine = FileEngine(FileConfig())
file_out = file_engine.process(FileInput(70, 1.0, 0.2, 1.0, 0.1))
print("Import-from-file check:", file_out)
nb_out = engine.process(RawImportanceInput(70, 1.0, 0.2, 1.0, 0.1))
assert abs(file_out.safe_importance - nb_out.safe_importance) < 1e-9
assert abs(file_out.base_importance - nb_out.base_importance) < 1e-9
print("File modules match the notebook implementation.")


## 18. Final Demonstration
One region traced through every stage. Every number is computed live.


In [ ]:
# Cell: Final Demonstration (all numbers computed live by the code).
final_region = RawImportanceInput(distance_m=70.0, semantic_importance=1.0,
                                  terrain_complexity=0.2, dynamic_relevance=1.0,
                                  uncertainty=0.1)
final_out = engine.process(final_region)
print("-" * 42)
print("Adaptive Resolution Decision")
print("-" * 42)
print(f"Distance:              {final_region.distance_m:>10.2f} m")
print(f"Distance score:        {final_out.distance_score:>10.3f}")
print(f"Semantic importance:   {final_out.semantic_score:>10.3f}")
print(f"Terrain complexity:    {final_out.terrain_score:>10.3f}")
print(f"Dynamic relevance:     {final_out.dynamic_score:>10.3f}")
print(f"Uncertainty:           {final_out.uncertainty_score:>10.3f}")
print(f"Base importance:       {final_out.base_importance:>10.3f}")
print(f"Safe importance:       {final_out.safe_importance:>10.3f}")
print(f"Selected resolution:   {final_out.selected_resolution_m:>10.2f} m ({final_out.resolution_level})")
print("-" * 42)


## 19. Integration Notes + Summary
**Future LiDAR interface.** The engines stay dataset-agnostic; only a thin converter is added later:

```
SemanticKITTI -> Preprocessing -> Semantic perception -> Terrain / geometric analysis
-> RegionFeatures (RawImportanceInput) -> Importance Engine -> Resolution Engine -> 2.5D mapper
```

Concretely: replace manual `RawImportanceInput(...)` construction with something like
`region = features_from_lidar_cluster(cluster, semantics, geometry)`. The engines themselves
do not change.

### Summary
1. **What the Importance Engine does:** fuses distance, semantic, terrain, dynamic, and
   uncertainty scores into `I_base` (weighted average) and then `I_safe` (uncertainty bonus).
   Output is always in [0,1].
2. **What the Resolution Engine does:** maps `I_safe` to 5 / 10 / 20 / 50 cm through the
   initial thresholds 0.75 / 0.50 / 0.25, so detail goes where it matters.
3. **Why normalization is needed:** it puts metres and unitless scores on one comparable
   [0,1] scale; without it the weighted sum would be meaningless.
4. **Why uncertainty is handled separately:** it is both a normal factor *and* a safety
   signal. The `lambda * U` bonus lets uncertain regions step up to finer resolution instead
   of risking lost information - and `lambda` can be A/B tested (`lambda = 0` disables it).
5. **How this connects to the real LiDAR pipeline later:** any perception stack that can
   produce the five region features plugs in via a small converter; the engines already
   speak `RawImportanceInput`.

**What this notebook did NOT do:** no optimized weights, no real-time claim, no novelty
claim - prototype engineering baseline only. Next step: experiments on real data.


In [ ]:
# Cell: Final Demonstration (all numbers computed live by the code).
print("Input: RegionFeatures -> Normalization -> Importance Engine -> score -> safety -> Resolution Engine -> 5/10/20/50cm")
final_region = RawImportanceInput(distance_m=70.0, semantic_importance=1.0,
                                  terrain_complexity=0.2, dynamic_relevance=1.0,
                                  uncertainty=0.1)
final_out = engine.process(final_region)
print("-" * 42)
print("Adaptive Resolution Decision")
print("-" * 42)
print(f"Distance:              {final_region.distance_m:>10.2f} m")
print(f"Distance score:        {final_out.distance_score:>10.3f}")
print(f"Semantic importance:   {final_out.semantic_score:>10.3f}")
print(f"Terrain complexity:    {final_out.terrain_score:>10.3f}")
print(f"Dynamic relevance:     {final_out.dynamic_score:>10.3f}")
print(f"Uncertainty:           {final_out.uncertainty_score:>10.3f}")
print(f"Base importance:       {final_out.base_importance:>10.3f}")
print(f"Safe importance:       {final_out.safe_importance:>10.3f}")
print(f"Selected resolution:   {final_out.selected_resolution_m:>10.2f} m ({final_out.resolution_level})")
print("-" * 42)
